In [ ]:
# run classical ML model on ref-alt matrix

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV, RidgeCV, LogisticRegressionCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.preprocessing import LabelEncoder


In [2]:

# --- PR Helper Functions ---
def compute_residue_scores_from_coef(coef, L):
    return np.abs(coef[:L])

def load_catalog(catalog_path, allowed_confidences):
    catalog = pd.read_csv(catalog_path)
    catalog = catalog[
        (catalog["confidence"].isin(allowed_confidences)) &
        (catalog["intersectional"] == True)
    ].copy()
    catalog["aa_pos_0idx"] = catalog["aa_pos"].astype(int) - 1
    return catalog


def evaluate_topk_precision_recall_refalt(gene_name, scores, catalog_df, k_vals=[10, 20, 30, 50, 100]):
    gene_catalog = catalog_df[catalog_df["gene"].str.lower() == gene_name.lower()].copy()
    if gene_catalog.empty:
        print(f"Skipping {gene_name}: no intersectional variants found.")
        return []

    total_positives = len(set(gene_catalog["aa_pos_0idx"]))

    imp_df = pd.DataFrame({"Residue_Position": np.arange(len(scores)), "Importance": scores})
    imp_df_sorted = imp_df.sort_values("Importance", ascending=False)

    rows = []
    for k in k_vals:
        top_k = imp_df_sorted.head(k)
        important_positions = set(top_k["Residue_Position"])
        matched_df = gene_catalog[gene_catalog["aa_pos_0idx"].isin(important_positions)]

        tp = matched_df["aa_pos_0idx"].nunique()
        prec = tp / k if k > 0 else 0
        rec = tp / total_positives if total_positives > 0 else 0
        f1 = 2 * prec * rec / (prec + rec + 1e-8)

        identified_variants = matched_df.drop_duplicates("aa_pos_0idx")["variant"].tolist()

        rows.append({
            "gene": gene_name, "model": None, "k": k,
            "TP": tp,
            "Total_Resistance_Positions": total_positives,
            "precision": prec,
            "recall": rec,
            "F1": f1,
            "Identified_Variants": ", ".join(identified_variants) if identified_variants else "None"
        })
    return rows


# --- Data Loaders ---
def load_feature_matrix_and_labels(gene_name):
    X = np.load(f"./data/feature_matrix_labels/{gene_name}_feature_matrix.npy")
    y = np.load(f"./data/feature_matrix_labels/{gene_name}_labels.npy", allow_pickle=True)
    return X, y

def encode_labels(y):
    le = LabelEncoder()
    return le.fit_transform(y)

# --- Main Training + Evaluation ---
def run_models(gene_name):
    print(f"\n=== Running models for {gene_name} ===")
    X, y = load_feature_matrix_and_labels(gene_name)
    y_encoded = encode_labels(y)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
    )

    ALLOWED_CONFIDENCES = ['1) Assoc w R', '2) Assoc w R - Interim']
    who_df = load_catalog("./data/filtered_variants_output.csv", ALLOWED_CONFIDENCES)

    L = X.shape[1]  # sequence length in residue-level encoding

    results = {}
    pr_all_models = []

    # ----- Lasso -----
    lasso = LassoCV(max_iter=10000, cv=5, random_state=42, alphas=[0.001, 0.01, 0.1, 1, 10, 100])
    lasso.fit(X_train, y_train)
    results['lasso'] = {
        'r2': lasso.score(X_test, y_test),
        'auc': roc_auc_score(y_test, lasso.predict(X_test)),
        'mse': mean_squared_error(y_test, lasso.predict(X_test))
    }
    # PR
    scores = compute_residue_scores_from_coef(lasso.coef_, L)
    pr_rows = evaluate_topk_precision_recall_refalt(gene_name, scores, who_df)
    for r in pr_rows:
        r["model"] = "lasso"
    pr_all_models.extend(pr_rows)

    # ----- Ridge -----
    ridge = RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10])
    ridge.fit(X_train, y_train)
    results['ridge'] = {
        'r2': ridge.score(X_test, y_test),
        'auc': roc_auc_score(y_test, ridge.predict(X_test)),
        'mse': mean_squared_error(y_test, ridge.predict(X_test))
    }
    scores = compute_residue_scores_from_coef(ridge.coef_, L)
    pr_rows = evaluate_topk_precision_recall_refalt(gene_name, scores, who_df)
    for r in pr_rows:
        r["model"] = "ridge"
    pr_all_models.extend(pr_rows)

    # ----- Logistic Regression -----
    log_model = LogisticRegressionCV(
        cv=3, scoring="roc_auc", max_iter=5000,
        Cs=[1e-4, 1e-3, 1e-2, 0.1, 1, 10, 100],
        class_weight="balanced"
    )
    log_model.fit(X_train, y_train)
    results['logreg'] = {
        'r2': log_model.score(X_test, y_test),
        'auc': roc_auc_score(y_test, log_model.predict(X_test)),
        'mse': mean_squared_error(y_test, log_model.predict(X_test))
    }
    scores = compute_residue_scores_from_coef(log_model.coef_[0], L)  # note: coef_ shape is (1, L)
    pr_rows = evaluate_topk_precision_recall_refalt(gene_name, scores, who_df)
    for r in pr_rows:
        r["model"] = "logreg"
    pr_all_models.extend(pr_rows)

    # ----- Output -----
    for model_name, metrics in results.items():
        print(f"\n[{model_name.upper()}] Results for {gene_name}:")
        for metric, val in metrics.items():
            print(f"{metric}: {val:.4f}")

    # Save PR summary
    pr_df = pd.DataFrame(pr_all_models)
    pr_df.to_csv(f"./residue_importance/ref_alt/pr_summary_refalt_{gene_name}.csv", index=False)

    return results


In [3]:
ref_seqs=pd.read_csv('./data/catalog/protein_sequences.csv')
gene_list=['rpoB','rpsL','tlyA','pncA','eis','gid','katG','inhA','embA','embB', 'embC','gyrB', 'gyrA', 'ethA', 'ethR']

In [4]:
all_results = {}
all_pr_rows = []
for gene in gene_list:
    try:
        print(f"\nRunning models for {gene}")
        result = run_models(gene)

        # Load the per-gene PR CSV just created
        pr_df = pd.read_csv(f"./residue_importance/ref_alt/pr_summary_refalt_{gene}.csv")
        all_pr_rows.append(pr_df)

        all_results[gene] = result
    except Exception as e:
        print(f"Error processing {gene}: {e}")


Running models for rpoB

=== Running models for rpoB ===

[LASSO] Results for rpoB:
r2: 0.8184
auc: 0.9583
mse: 0.0321

[RIDGE] Results for rpoB:
r2: 0.8346
auc: 0.9628
mse: 0.0292

[LOGREG] Results for rpoB:
r2: 0.9626
auc: 0.9569
mse: 0.0311

Running models for rpsL

=== Running models for rpsL ===

[LASSO] Results for rpsL:
r2: 0.4353
auc: 0.7739
mse: 0.1205

[RIDGE] Results for rpsL:
r2: 0.4354
auc: 0.7739
mse: 0.1205

[LOGREG] Results for rpsL:
r2: 0.7739
auc: 0.7211
mse: 0.1734

Running models for tlyA

=== Running models for tlyA ===

[LASSO] Results for tlyA:
r2: -0.0000
auc: 0.5000
mse: 0.1173

[RIDGE] Results for tlyA:
r2: 0.0052
auc: 0.5024
mse: 0.1166

[LOGREG] Results for tlyA:
r2: 0.5024
auc: 0.5052
mse: 0.1343

Running models for pncA

=== Running models for pncA ===

[LASSO] Results for pncA:
r2: 0.1448
auc: 0.6239
mse: 0.0949

[RIDGE] Results for pncA:
r2: 0.4324
auc: 0.8037
mse: 0.0630

[LOGREG] Results for pncA:
r2: 0.8352
auc: 0.6363
mse: 0.0974

Running models for

In [8]:
# Combine all per-gene PR summaries
all_pr_df = pd.concat(all_pr_rows, ignore_index=True)
all_pr_df.to_csv("residue_importance/pr_summary_refalt_all_genes.csv", index=False)
print("Combined PR summary saved to pr_summary_refalt_all_genes.csv")

Combined PR summary saved to pr_summary_refalt_all_genes.csv


In [10]:
# Sort by 'gene' ascending and 'precision' descending
df_sorted = all_pr_df.sort_values(by=['gene', 'precision'], ascending=[True, False])

# Now, for each gene, pick the first row (highest precision per gene)
best_per_gene = df_sorted.drop_duplicates(subset=['gene'], keep='first').reset_index(drop=True)

print(best_per_gene)

    gene   model    k  TP  Total_Resistance_Positions  precision    recall  \
0   embB   lasso   10   1                           6       0.10  0.166667   
1   ethA   ridge  100   3                           9       0.03  0.333333   
2    gid  logreg   20   3                           8       0.15  0.375000   
3   gyrA   ridge   10   2                           5       0.20  0.400000   
4   gyrB   ridge   10   1                           5       0.10  0.200000   
5   inhA   lasso   10   0                           1       0.00  0.000000   
6   katG   lasso   10   0                           2       0.00  0.000000   
7   pncA  logreg   20  13                          95       0.65  0.136842   
8   rpoB   ridge   10   5                          26       0.50  0.192308   
9   rpsL   lasso   10   1                           2       0.10  0.500000   
10  tlyA   lasso   20   2                           2       0.10  1.000000   

          F1                                Identified_Variants

In [11]:
best_per_gene.to_csv("residue_importance/refalt_pr_overlap_summary.csv", index=False)

In [6]:
#Flatten it into a list of rows
rows = []
for gene, models in all_results.items():
    for model, metrics in models.items():
        row = {'gene': gene, 'model': model}
        row.update(metrics)
        rows.append(row)

In [7]:
# Convert to DataFrame
df = pd.DataFrame(rows)
df

,gene,model,r2,auc,mse
0,rpoB,lasso,0.818375,0.958284,0.032099
1,rpoB,ridge,0.834559,0.962801,0.029239
2,rpoB,logreg,0.962631,0.956946,0.031139
3,rpsL,lasso,0.435267,0.773858,0.120502
4,rpsL,ridge,0.435422,0.773858,0.120469
5,rpsL,logreg,0.773858,0.721115,0.173415
6,tlyA,lasso,-0.000001,0.500000,0.117260
7,tlyA,ridge,0.005227,0.502377,0.116647
8,tlyA,logreg,0.502377,0.505155,0.134266
9,pncA,lasso,0.144806,0.623855,0.094918


In [8]:
# Optional: rearrange columns
cols = ['gene', 'model'] + [c for c in df.columns if c not in ['gene', 'model']]
df = df[cols]

# View result
print(df.head())

   gene   model        r2       auc       mse
0  rpoB   lasso  0.818375  0.958284  0.032099
1  rpoB   ridge  0.834559  0.962801  0.029239
2  rpoB  logreg  0.962631  0.956946  0.031139
3  rpsL   lasso  0.435267  0.773858  0.120502
4  rpsL   ridge  0.435422  0.773858  0.120469


In [9]:
df.to_csv("auc_results/all_genes_regression_models_performance.csv", index=False)